### Data Engineering Assistant

#### Building a dense vector retrieval application using **LangChain**, **Gemini Embeddings**, and **Qdrant** as an in-memory vector database.

The goal is to understand the core RAG loop:

1. Load the Big Book of Data Engineering PDF
2. Split it into smaller chunks
3. Embed those chunks
4. Store the embeddings in Qdrant
5. Retrieve relevant chunks for a question
6. Generate an answer grounded in the retrieved context

In [2]:
#import warnings
#warnings.filterwarnings("ignore")

### 🛠️ Environmental Setup

In this step, we install `uv`, create a virtual environment, prepare a `requirements.txt` file, and install the Python packages needed for the notebook.

### Step 1: Install `uv`
Use the official installer to install `uv` on your system.

**Windows:**
```bash
powershell -ExecutionPolicy ByPass -c "irm [https://astral.sh/uv/install.ps1](https://astral.sh/uv/install.ps1) | iex"

**macOS and Linux:**
```bash
curl -LsSf [https://astral.sh/uv/install.sh](https://astral.sh/uv/install.sh) | sh
```

### Step 2: Check the installation
Open a new terminal and verify that `uv` is available.
```bash
uv --version
```
### Initialize a new project (this creates a pyproject.toml and standard structure)
```bash
uv init
```
### Step 3: Create a virtual environment
Create and activate the project environment before installing dependencies.
```bash
uv venv
.venv/bin/activate
```

### Step 4: Add the dependencies
Create a `requirements.txt` file with the packages required for this project.
```bash
langchain-core
langchain-community
langchain-text-splitters
langchain-google-genai
langchain-qdrant
qdrant-client
python-dotenv
ipykernel
jupyter
pypdf
```

### Step 5: Install the packages
Use `uv` to install everything from `requirements.txt`.
```bash
uv pip install -r requirements.txt
```

### Imports

LangChain v1 separates integrations into partner packages. We will use:

- `langchain_community` for PDF loading
- `langchain_text_splitters` for chunking
- `langchain_google_genai` for chat and embedding models
- `langchain_qdrant` for the Qdrant vector store

In [20]:
import os
from dotenv import load_dotenv
from pathlib import Path
import logging
from pathlib import Path
from typing import List, Union, Optional, Dict

from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

An embedding model turns a piece of text into a numeric vector. If two pieces of text are related, their vectors should be close to each other. We measure that closeness with cosine similarity, which tells us how aligned two vectors are.

Why this matters
In RAG, we embed each document chunk once, then embed the user’s question and search for the most similar chunks. That retrieval step is what helps the model answer from the right part of the PDF.

In [14]:
from dotenv import load_dotenv
load_dotenv()

gemini_api_key = os.getenv("GEMINI_API_KEY")

### Understanding Embedding Similarity (Cosine Similarity)

Before working with a full PDF, it helps to understand the core idea behind vector-based retrieval: how a computer decides that two pieces of text are "similar."

### What an embedding is
An embedding model converts a piece of text into a list of numbers, called a **vector**. Each vector represents the meaning of the text in a high-dimensional space. Texts that are related in meaning end up with vectors that are close to each other in that space, while unrelated texts end up farther apart.

### How we measure "closeness"
The most common way to measure how close two vectors are is **cosine similarity**. Instead of measuring the straight-line distance between two vectors, cosine similarity looks at the **angle** between them.

The formula is:

$$
\text{cosine similarity}(a, b) = \frac{a \cdot b}{\|a\| \, \|b\|}
$$

Where:
- $a \cdot b$ is the dot product of the two vectors.
- $\|a\|$ and $\|b\|$ are the magnitudes (lengths) of vectors a and b.

### How to interpret the score
- A score close to **1** means the vectors point in almost the same direction, so the texts are considered highly similar.
- A score close to **0** means the vectors are unrelated (orthogonal), so the texts are not similar.
- A score close to **-1** means the vectors point in opposite directions, meaning the texts may be conceptually opposite.

### Why this matters for retrieval
This is exactly the mechanism vector databases use, just applied at a much larger scale across thousands or millions of document chunks. When a user asks a question, the question itself is converted into a vector, and the database finds the document chunks whose vectors are most similar to that question vector.

### Key takeaway
Cosine similarity does not tell you whether an answer is *factually correct* — it only tells you that two pieces of text are *related in meaning*. It is a **relevance signal**, not a **truth signal**. This ranking is what allows a Retrieval-Augmented Generation (RAG) system to find the most relevant chunks of a document before generating an answer.

In [21]:
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def load_pdf_documents(
    file_path: Union[str, Path], 
    extra_metadata: Optional[Dict[str, str]] = None
) -> List[Document]:
    """Loads a PDF, filters out empty pages, and injects custom metadata."""
    pdf_path = Path(file_path)

    if not pdf_path.exists():
        raise FileNotFoundError(
            f"Expected a PDF at: {pdf_path.resolve()}\n"
            "Please verify the file path."
        )

    loader = PyPDFLoader(str(pdf_path))
    valid_pages = []
    
    # Safely handle the case where no extra metadata is provided
    extra_metadata = extra_metadata or {}

    for page in loader.lazy_load():
        if not page.page_content.strip():
            continue  
            
        # 1. Update with a default source (always useful for tracking)
        page.metadata.update({"source": pdf_path.name})
        
        # 2. Inject whatever custom metadata the user passed in
        page.metadata.update(extra_metadata)
        
        valid_pages.append(page)

    if not valid_pages:
        raise ValueError(
            f"The PDF '{pdf_path.name}' loaded, but no extractable text was found. "
            "This usually means the PDF is scanned and needs OCR first."
        )

    logger.info(f"Loaded {len(valid_pages)} text-containing pages from {pdf_path.name}.")
    
    return valid_pages

In [22]:
pages = load_pdf_documents("Data/Big_Book_Of_Data_Engineering.pdf")

INFO:__main__:Loaded 127 text-containing pages from Big_Book_Of_Data_Engineering.pdf.


In [27]:
print(pages[12].page_content[:7500])
print("\nMetadata:", pages[0].metadata)

Conclusion  
As organizations strive to innovate with AI, data engineering is a focal point for 
success by delivering reliable, real-time data pipelines that make AI possible. 
With the Databricks Platform, built on a lakehouse architecture and powered by 
Data Intelligence, data engineers are set up for success in dealing with the critical 
challenges posed in the modern data landscape. By leaning on the advanced 
capabilities of the Data Intelligence Platform, data engineers don’t need to spend 
as much time managing complex pipelines or dealing with reliability, scalability 
and data quality issues. Instead, they can focus on innovation and bringing more 
value to the organization.
FOLLOW PROVEN BEST PRACTICES 
In the next section, we describe best practices for data engineering and  
end-to-end use cases drawn from real-world examples. From data ingestion  
and real-time processing to orchestration and data federation, you’ll learn how 
to apply pr
oven patterns and make the best 

### Chunking the Document

A full PDF page can be too large or cover too many mixed topics for high-quality retrieval. To fix this, we split each page into smaller, overlapping chunks so that each chunk stays focused but still keeps enough local context.

We will start with chunks of **1,000 characters** and **200 characters of overlap**.

- **Chunk size**: controls how much text each vector represents.
- **Chunk overlap**: keeps nearby context from being lost at chunk boundaries.

`RecursiveCharacterTextSplitter` tries to split on natural boundaries first, such as paragraphs and line breaks, before falling back to smaller separators like sentences or words.

In [32]:
def chunk_documents(
    documents: List[Document], 
    chunk_size: int = 1000, 
    chunk_overlap: int = 200
) -> List[Document]:
    """Splits a list of documents into chunks and prints a preview of the first one."""
    
    if not documents:
        logger.warning("No documents were provided to split. Returning an empty list.")
        return []

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        add_start_index=True,
    )

    splits = text_splitter.split_documents(documents)

    logger.info(
        f"Split {len(documents)} pages into {len(splits)} chunks "
        f"(Size: {chunk_size}, Overlap: {chunk_overlap})."
    )

    # --- Inspection logic added directly here ---
    if splits:
        sample_chunk = splits[0]
        print("\n--- Sample Chunk [0] Preview ---")
        print(sample_chunk.page_content[:750])
        print("\nMetadata:", sample_chunk.metadata)
        print("--------------------------------\n")

    return splits

In [33]:
chunks = chunk_documents(pages, chunk_size=1000, chunk_overlap=200)

INFO:__main__:Split 127 pages into 332 chunks (Size: 1000, Overlap: 200).



--- Sample Chunk [0] Preview ---
sales@3cloudsolutions.com // 888-88-AZURE // 3cloudsolutions.com

Metadata: {'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 20.1 (Windows)', 'creationdate': '2025-01-24T01:32:23+00:00', 'moddate': '2025-10-08T18:49:08-05:00', 'trapped': '/False', 'source': 'Big_Book_Of_Data_Engineering.pdf', 'total_pages': 127, 'page': 0, 'page_label': '1', 'start_index': 0}
--------------------------------



### Chunk Size and Chunk Overlap

**Chunk size** sets how much text goes into each chunk. **Chunk overlap** sets how much text repeats between one chunk and the next, so context isn't lost at the cut-off point.

**Choosing chunk size**
- Too small → chunks miss context and feel fragmented.
- Too large → chunks mix multiple ideas, making retrieval less focused.

**Choosing chunk overlap**
- Too little → key details near chunk edges can get lost.
- Too much → chunks repeat the same content, wasting storage and slowing retrieval.

In short, it's a balance between **precision and context**: smaller chunks with less overlap retrieve more precisely but may lack surrounding detail, while larger chunks with more overlap preserve context but retrieve less precisely. The best settings depend on how dense and technical your source document is.

### Embeddings and Qdrant

Now we apply the same embedding idea to every chunk from the PDF. Qdrant stores those vectors and lets us search for chunks that are close to a query in embedding space.

We already created a Gemini embedding model earlier. The Qdrant collection name is just a label for the set of vectors we are creating.

For this notebook, Qdrant runs in memory with `location="memory"`. This means no Docker, no Qdrant Cloud account, and no persistence after the notebook kernel stops.

In [41]:
from tenacity import retry, wait_fixed, stop_after_attempt

class SafeGoogleEmbeddings(GoogleGenerativeAIEmbeddings):
    
    # Changed to wait exactly 60 seconds, and gave it a generous 10 attempts
    @retry(wait=wait_fixed(60), stop=stop_after_attempt(10))
    def embed_documents(self, texts, *args, **kwargs):
        print(f"Embedding batch of {len(texts)} chunks...")
        return super().embed_documents(texts, *args, **kwargs)

# 1. Initialize your new safe class
embedding_model = "models/gemini-embedding-001"
embeddings = SafeGoogleEmbeddings(model=embedding_model)

collection_name = "data_engineering_ebook"

# 2. Run the pipeline
print("Starting vector store creation (this may take a few minutes if rate limited)...")

vector_store = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    location=":memory:",
    collection_name=collection_name,
    force_recreate=True,
)

print(f"Embedded chunks with Gemini embedding model")
print(f"Built in-memory Qdrant collection: {collection_name}")

Starting vector store creation (this may take a few minutes if rate limited)...
Embedding batch of 1 chunks...


INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"


Embedding batch of 64 chunks...


INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"


Embedding batch of 64 chunks...


...

Embedding batch of 64 chunks...


INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"


Embedding batch of 12 chunks...


INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"


Embedded chunks with Gemini embedding model
Built in-memory Qdrant collection: data_engineering_ebook


In [42]:
# 1. Check the raw collection stats
# We can access the underlying Qdrant client directly from LangChain
qdrant_client = vector_store.client
collection_info = qdrant_client.get_collection(collection_name)

print("--- Database Stats ---")
print(f"Total chunks successfully stored: {collection_info.points_count}")
print("-" * 22)


# 2. Test the Semantic Search
# Ask a question that you know is covered in the Data Engineering book
test_query = "What are the core responsibilities of a data engineer?"
print(f"\nSearching for: '{test_query}'\n")

# k=2 tells it to bring back the top 2 most relevant chunks
results = vector_store.similarity_search(test_query, k=2)

if not results:
    print("No results found. Something went wrong!")
else:
    for i, doc in enumerate(results, 1):
        # Print the metadata and a snippet of the text
        source = doc.metadata.get("source", "Unknown")
        print(f"--- Result {i} (From: {source}) ---")
        
        # Print the first 300 characters to keep the terminal clean
        preview = doc.page_content[:300].replace('\n', ' ')
        print(f"{preview}...\n")

--- Database Stats ---
Total chunks successfully stored: 332
----------------------

Searching for: 'What are the core responsibilities of a data engineer?'



INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"


--- Result 1 (From: Big_Book_Of_Data_Engineering.pdf) ---
factory floors, more and more data is created and streamed in real time and requires low-latency processing so it can be used in real-time decision-making. ■ Scaling data pipelines reliably:  With data coming in large quantities and often in real time, scaling the compute infrastructure that runs da...

--- Result 2 (From: Big_Book_Of_Data_Engineering.pdf) ---
Challenges of data engineering in the AI era As previously mentioned, data engineering is key to ensuring reliable data for  AI initiatives. Data engineers who build and maintain ETL pipelines and the  data infrastructure that underpins analytics and AI workloads face specific  challenges in this fa...

